# M7 · LoRA fine-tune of the MarketingOS domain classifier

Distils the production idea classifier (`normalize_idea()`, teacher: `gpt-oss-120b` on Groq) into
**Qwen2.5-1.5B-Instruct** with a LoRA adapter, then exports a 4-bit **GGUF** the app runs on CPU.

**Run on Colab with a T4 GPU:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.
Takes roughly 20–40 minutes. You'll be asked to upload the dataset files in step 2.

## 1 · Install

In [ ]:
%%capture
!pip install unsloth

## 2 · Upload the dataset

Upload from `src/finetune/data/` in the repo: **`train.jsonl`**, **`val.jsonl`**, **`test.jsonl`**, and optionally
**`real_campaigns.jsonl`** (git-ignored real campaigns, from `python -m src.finetune.export_dataset`).
Real campaigns are added to training only.

In [ ]:
import json
from google.colab import files

uploaded = files.upload()

def load(name):
    if name not in uploaded:
        return []
    return [json.loads(line) for line in uploaded[name].decode('utf-8').splitlines() if line.strip()]

train_rows = load('train.jsonl') + load('real_campaigns.jsonl')
val_rows, test_rows = load('val.jsonl'), load('test.jsonl')
print(f'train {len(train_rows)} | val {len(val_rows)} | test {len(test_rows)}')
assert train_rows and val_rows and test_rows, 'train.jsonl, val.jsonl and test.jsonl are required'

## 3 · Prompt (must match `src/finetune/prompt.py` exactly)

In [ ]:
CATEGORIES = ['fitness_wellness', 'b2b_saas', 'ecommerce_retail', 'food_beverage', 'finance_fintech', 'education', 'consumer_electronics', 'travel_hospitality', 'real_estate', 'healthcare', 'entertainment_media', 'other']
SYSTEM = 'You classify ad-campaign ideas into one product category. Answer with the category name only.'

def build_messages(idea):
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f"Categories: {', '.join(CATEGORIES)}\nIdea: {idea.strip()}\nCategory:"},
    ]

def parse_label(text):
    cleaned = text.strip().strip('"\'`.').lower()
    if cleaned in CATEGORIES:
        return cleaned
    return next((c for c in CATEGORIES if c in cleaned), None)

## 4 · Load the base model and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

BASE = 'unsloth/Qwen2.5-1.5B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=512, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=7,
)

## 5 · Format examples (the answer is the bare label)

In [ ]:
from datasets import Dataset

def to_text(row):
    messages = build_messages(row['idea']) + [{'role': 'assistant', 'content': row['label']}]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(train_rows).map(to_text)
val_ds = Dataset.from_list(val_rows).map(to_text)
print(train_ds[0]['text'])

## 6 · Train (loss only on the answer tokens)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,  # older TRL versions call this `tokenizer=`
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=512,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='linear',
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy='epoch',
        seed=7,
        output_dir='outputs',
        report_to='none',
    ),
)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

## 7 · Accuracy on the held-out test set

In [ ]:
import time
from collections import Counter

FastLanguageModel.for_inference(model)

def classify(idea):
    prompt = tokenizer.apply_chat_template(build_messages(idea), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    return parse_label(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

start, hits, invalid, confusions = time.time(), 0, 0, Counter()
for row in test_rows:
    pred = classify(row['idea'])
    invalid += pred is None
    hits += pred == row['label']
    if pred != row['label']:
        confusions[(row['label'], pred)] += 1
print(f'test accuracy {hits / len(test_rows):.1%} ({hits}/{len(test_rows)}) | invalid outputs {invalid} | {(time.time() - start) / len(test_rows) * 1000:.0f} ms/idea on GPU')
print('most common confusions (expected -> predicted):', confusions.most_common(5))

## 8 · Export

Saves the LoRA adapter and a 4-bit **GGUF** (≈1 GB) for CPU inference in the app, then downloads them.
Put the `.gguf` in `src/finetune/models/` locally (git-ignored); the integration step will publish it for the workers.

In [ ]:
import glob, shutil

model.save_pretrained('marketingos-domain-lora')
tokenizer.save_pretrained('marketingos-domain-lora')
shutil.make_archive('marketingos-domain-lora', 'zip', 'marketingos-domain-lora')

model.save_pretrained_gguf('marketingos-domain-gguf', tokenizer, quantization_method='q4_k_m')
gguf = sorted(glob.glob('marketingos-domain-gguf*/*.gguf') + glob.glob('*.gguf'))
print('GGUF files:', gguf)

files.download('marketingos-domain-lora.zip')
for path in gguf:
    files.download(path)